# Llama-2-7B — INT8-Simulated Norm Proxy Experiment

This notebook tests whether the Norm Proxy selector remains valid when Q/K are quantized to INT8 before block-norm calculation.

It measures **FP-vs-INT8 proxy ranking**, **block-selection overlap**, and **LongBench quality** at 50/70/80/90% sparsity.

**Important:** the current project RTL specifies INT8 Q/K, but no final activation-scale register/method is defined in the available RTL context. Therefore the notebook uses an explicit symmetric per-head dynamic scale. Replace that scale function when the final hardware Q/K scale is fixed.

The quality experiment uses INT8 only for the **selector**, while exact selected QK remains in the original floating activation precision. This isolates the effect of INT8 quantization on the pruning decision.


In [1]:

# ============================================================
# CELL 1 — SETUP / MODEL / LONGBENCH
# ============================================================

!pip install -q transformers accelerate bitsandbytes sentencepiece datasets==2.21.0 rouge_score huggingface_hub

import os
import sys
import json
import math
import gc
import types
import re
import string
import subprocess
from collections import defaultdict, Counter

import torch
import pandas as pd

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from transformers.models.llama.modeling_llama import (
    LlamaAttention,
    apply_rotary_pos_emb,
)

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"

SEQ_LEN = 2048

TASKS = [
    "narrativeqa",
    "qasper",
    "gov_report",
]

SAMPLES_PER_TASK = 15
MAX_TOTAL_SAMPLES = 45

BLOCK_ROWS = 32
BLOCK_COLS = 32

# Official Dense baseline from the completed experiment.
DENSE_AVERAGE_SCORE = 11.8900

DENSE_TASK_SCORES = {
    "gov_report": 34.17,
    "narrativeqa": 0.42,
    "qasper": 1.08,
}

print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# LongBench repository
# ------------------------------------------------------------

if not os.path.exists("/content/LongBench"):
    !git clone -q https://github.com/THUDM/LongBench.git /content/LongBench
else:
    print("LongBench already exists.")

# ------------------------------------------------------------
# Hugging Face login
# ------------------------------------------------------------

from huggingface_hub import login
login()

# ------------------------------------------------------------
# Model
#
# Same model/loading setup used by the previous experiment:
# Llama-2-7B, 4-bit weights, FP16 compute.
#
# Important:
#   The Q/K ACTIVATIONS produced by the model are FP16.
#   The new experiment explicitly quantizes those Q/K
#   activations to INT8 for the proxy selector.
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
)

model.eval()

print("Model loaded:", MODEL_NAME)
print("Attention implementation: eager")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.6.1 which is incompatible.
GPU: Tesla T4


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Model loaded: meta-llama/Llama-2-7b-chat-hf
Attention implementation: eager


In [2]:

# ============================================================
# CELL 2 — LOAD LONG BENCH
# ============================================================

CONFIG_DIR = "/content/LongBench/LongBench/config"

with open(
    f"{CONFIG_DIR}/dataset2prompt.json",
    "r",
    encoding="utf-8",
) as f:
    dataset2prompt = json.load(f)

maxlen_path = f"{CONFIG_DIR}/dataset2maxlen.json"

if os.path.exists(maxlen_path):
    with open(
        maxlen_path,
        "r",
        encoding="utf-8",
    ) as f:
        dataset2maxlen = json.load(f)
else:
    dataset2maxlen = {}

samples = []

for task in TASKS:

    print("=" * 60)
    print("Loading:", task)

    ds = load_dataset(
        "THUDM/LongBench",
        task,
        split="test",
    )

    take = min(
        SAMPLES_PER_TASK,
        len(ds),
    )

    for item in ds.select(
        range(take)
    ):

        samples.append({
            "task": task,
            "context": item["context"],
            "input": item["input"],
            "answers": item["answers"],
            "all_classes": item.get(
                "all_classes",
                [],
            ),
            "length": item.get(
                "length",
                None,
            ),
        })

samples = samples[
    :MAX_TOTAL_SAMPLES
]

print("=" * 60)
print("TOTAL SAMPLES:", len(samples))
print(
    "BY TASK:",
    Counter(
        s["task"]
        for s in samples
    ),
)

def generate_longbench_answer(sample):

    task = sample["task"]

    prompt = dataset2prompt[
        task
    ].format(
        **sample
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=SEQ_LEN,
    ).to(model.device)

    max_new_tokens = int(
        dataset2maxlen.get(
            task,
            64,
        )
    )

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True,
    )


def run_longbench_eval(model_name):

    cmd = [
        sys.executable,
        "/content/LongBench/LongBench/eval.py",
        "--model",
        model_name,
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    print(result.stdout)

    if result.stderr:
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"LongBench evaluation failed: {model_name}"
        )

    return result.stdout

print("LongBench helper ready.")


Loading: narrativeqa


The repository for THUDM/LongBench contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/THUDM/LongBench.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating test split: 0 examples [00:00, ? examples/s]

Loading: qasper


Generating test split: 0 examples [00:00, ? examples/s]

Loading: gov_report


Generating test split: 0 examples [00:00, ? examples/s]

TOTAL SAMPLES: 45
BY TASK: Counter({'narrativeqa': 15, 'qasper': 15, 'gov_report': 15})
LongBench helper ready.


In [3]:

# ============================================================
# CELL 3 — DEVICE + PATCH
# ============================================================

device = next(
    model.parameters()
).device

print("Model device:", device)

# Paste/execute the original vectorized Norm-Proxy attention
# implementation used in the previous experiment.
#
# The next cell overrides get_norm_proxy_mask() with the
# INT8-simulated selector.


Model device: cuda:0


In [7]:
# ============================================================
# CELL 3B — NORM PROXY + VECTORIZED ATTENTION PATCH
# ============================================================
#
# STANDALONE NORM-PROXY BRANCH
#
# Methodology:
#
#   Q/K
#    ↓
#   RoPE
#    ↓
#   INT8 quantization
#    ↓
#   ||Q_block||F × ||K_block||F
#    ↓
#   Top-K proxy blocks
#    ↓
#   EXACT QK ONLY FOR SELECTED BLOCKS
#    ↓
#   causal mask
#    ↓
#   softmax
#    ↓
#   attention × V
#
# NO L1(QK)
# NO L1 ranking
# NO full QK^T computation
#
# Block size:
#   32 × 32
#
# ============================================================

import math
import types
import torch
import torch.nn.functional as F

from transformers.models.llama.modeling_llama import LlamaAttention


# ============================================================
# CONFIGURATION
# ============================================================

BLOCK_ROWS = 32
BLOCK_COLS = 32

CURRENT_PROXY_SPARSITY = 50.0


# ============================================================
# GLOBAL STATISTICS
# ============================================================

proxy_stats = {}


def reset_proxy_stats():

    global proxy_stats

    proxy_stats = {
        "candidate_blocks": 0,
        "kept_blocks": 0,
        "threshold_sum": 0.0,
        "threshold_count": 0,
        "attention_calls": 0,
    }


reset_proxy_stats()


# ============================================================
# INT8 SYMMETRIC QUANTIZATION
# ============================================================

def quantize_int8_symmetric(x):
    """
    Symmetric per-tensor INT8 quantization.

    x:
        arbitrary floating-point tensor

    returns:
        q8     : int8 tensor
        scale  : floating-point scale
    """

    max_abs = x.abs().amax()

    if not torch.isfinite(max_abs):
        raise RuntimeError(
            "Non-finite value encountered during INT8 quantization."
        )

    if max_abs.item() == 0:
        scale = torch.tensor(
            1.0,
            device=x.device,
            dtype=x.dtype,
        )

    else:
        scale = max_abs / 127.0

    q8 = torch.clamp(
        torch.round(x / scale),
        -127,
        127,
    ).to(torch.int8)

    return q8, scale


# ============================================================
# BLOCK NORM PROXY
# ============================================================

def get_norm_proxy_mask(
    q,
    k,
    target_sparsity,
    block_rows=32,
    block_cols=32,
):
    """
    Generate block KEEP mask using ONLY:

        Proxy(Bij) = ||Q_i||F × ||K_j||F

    q:
        [Q_len, head_dim]

    k:
        [K_len, head_dim]

    IMPORTANT:
        This function does NOT compute QK^T.
    """

    global proxy_stats

    q_len, head_dim = q.shape
    k_len, _ = k.shape

    # --------------------------------------------------------
    # Only complete 32x32 blocks are candidates.
    # --------------------------------------------------------

    num_q_blocks = q_len // block_rows
    num_k_blocks = k_len // block_cols

    keep_mask = torch.zeros(
        num_q_blocks,
        num_k_blocks,
        dtype=torch.bool,
        device=q.device,
    )

    if (
        num_q_blocks == 0
        or num_k_blocks == 0
    ):
        return (
            keep_mask,
            0,
            0,
            float("nan"),
        )

    # --------------------------------------------------------
    # INT8 quantization
    # --------------------------------------------------------

    q8, q_scale = quantize_int8_symmetric(q)
    k8, k_scale = quantize_int8_symmetric(k)

    # --------------------------------------------------------
    # Reshape into blocks
    #
    # Q:
    # [Q_len, D]
    #      ↓
    # [num_q_blocks, 32, D]
    #
    # K:
    # [K_len, D]
    #      ↓
    # [num_k_blocks, 32, D]
    # --------------------------------------------------------

    q8_blocks = q8[
        :num_q_blocks * block_rows
    ].reshape(
        num_q_blocks,
        block_rows,
        head_dim,
    )

    k8_blocks = k8[
        :num_k_blocks * block_cols
    ].reshape(
        num_k_blocks,
        block_cols,
        head_dim,
    )

    # --------------------------------------------------------
    # Frobenius norms
    #
    # ||Q_i||F
    # ||K_j||F
    #
    # INT8 values are used here.
    # Scale is common and therefore does not affect ranking
    # when the same scale is used across all blocks.
    # --------------------------------------------------------

    q_norm = torch.sqrt(
        (q8_blocks.float() ** 2).sum(
            dim=(1, 2)
        )
    )

    k_norm = torch.sqrt(
        (k8_blocks.float() ** 2).sum(
            dim=(1, 2)
        )
    )

    # --------------------------------------------------------
    # Proxy matrix
    #
    # [num_q_blocks, 1]
    #       ×
    # [1, num_k_blocks]
    #
    # =
    #
    # [num_q_blocks, num_k_blocks]
    # --------------------------------------------------------

    proxy_scores = (
        q_norm[:, None]
        * k_norm[None, :]
    )

    # --------------------------------------------------------
    # Causal block constraint
    #
    # Query block i may only attend to
    # key blocks j <= i.
    # --------------------------------------------------------

    block_i = torch.arange(
        num_q_blocks,
        device=q.device,
    )[:, None]

    block_j = torch.arange(
        num_k_blocks,
        device=q.device,
    )[None, :]

    causal_blocks = (
        block_j <= block_i
    )

    # --------------------------------------------------------
    # Count candidate blocks
    # --------------------------------------------------------

    candidate_count = int(
        causal_blocks.sum().item()
    )

    # --------------------------------------------------------
    # Select top-K proxy blocks
    #
    # target_sparsity = percentage PRUNED
    #
    # Example:
    #
    # 90% sparsity
    # → keep 10%
    # → top 10% proxy blocks
    # --------------------------------------------------------

    keep_ratio = (
        1.0
        - target_sparsity / 100.0
    )

    kept_count_total = 0
    threshold_values = []

    for qi in range(num_q_blocks):

        valid_keys = torch.where(
            causal_blocks[qi]
        )[0]

        n_candidates = int(
            valid_keys.numel()
        )

        if n_candidates == 0:
            continue

        n_keep = max(
            1,
            int(
                math.ceil(
                    n_candidates
                    * keep_ratio
                )
            ),
        )

        n_keep = min(
            n_keep,
            n_candidates,
        )

        scores = proxy_scores[
            qi,
            valid_keys,
        ]

        top_values, top_indices = torch.topk(
            scores,
            k=n_keep,
            largest=True,
            sorted=False,
        )

        selected_keys = valid_keys[
            top_indices
        ]

        keep_mask[
            qi,
            selected_keys
        ] = True

        kept_count_total += n_keep

        threshold_values.append(
            top_values.min()
        )

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    proxy_stats[
        "candidate_blocks"
    ] += candidate_count

    proxy_stats[
        "kept_blocks"
    ] += kept_count_total

    if len(threshold_values) > 0:

        threshold_tensor = torch.stack(
            threshold_values
        )

        proxy_stats[
            "threshold_sum"
        ] += float(
            threshold_tensor.sum().item()
        )

        proxy_stats[
            "threshold_count"
        ] += int(
            threshold_tensor.numel()
        )

        threshold = float(
            threshold_tensor.mean().item()
        )

    else:

        threshold = float("nan")

    return (
        keep_mask,
        candidate_count,
        kept_count_total,
        threshold,
    )


# ============================================================
# ROTARY EMBEDDING HELPER
# ============================================================

def _apply_llama_rope(
    module,
    query_states,
    key_states,
    position_ids,
    position_embeddings=None,
):
    """
    Handle both older and newer Transformers Llama
    rotary embedding APIs.
    """

    # --------------------------------------------------------
    # Newer Transformers:
    #
    # position_embeddings = (cos, sin)
    # --------------------------------------------------------

    if position_embeddings is not None:

        cos, sin = position_embeddings

        from transformers.models.llama.modeling_llama import (
            apply_rotary_pos_emb,
        )

        return apply_rotary_pos_emb(
            query_states,
            key_states,
            cos,
            sin,
        )

    # --------------------------------------------------------
    # Older Transformers:
    #
    # rotary_emb(hidden_states, position_ids)
    # --------------------------------------------------------

    rotary_emb = module.rotary_emb

    try:

        cos, sin = rotary_emb(
            query_states,
            position_ids,
        )

    except TypeError:

        cos, sin = rotary_emb(
            query_states,
            position_ids=position_ids,
        )

    from transformers.models.llama.modeling_llama import (
        apply_rotary_pos_emb,
    )

    return apply_rotary_pos_emb(
        query_states,
        key_states,
        cos,
        sin,
    )


# ============================================================
# SAVE ORIGINAL FORWARD
# ============================================================

_original_llama_attention_forward = (
    LlamaAttention.forward
)


# ============================================================
# RESTORE ORIGINAL FORWARD
# ============================================================

def restore_original_llama_attention():

    LlamaAttention.forward = (
        _original_llama_attention_forward
    )

    print(
        "Restored original "
        "LlamaAttention.forward."
    )


# ============================================================
# REPLACE _norm_proxy_forward()
# ============================================================

def _norm_proxy_forward(
    self,
    hidden_states,
    position_embeddings=None,
    attention_mask=None,
    position_ids=None,
    past_key_value=None,
    cache_position=None,
    **kwargs,
):

    global proxy_stats
    global CURRENT_PROXY_SPARSITY

    bsz, q_len, _ = hidden_states.shape

    proxy_stats["attention_calls"] += 1

    # ========================================================
    # GET HEAD CONFIG
    # ========================================================

    num_heads = int(
        self.config.num_attention_heads
    )

    num_key_value_heads = int(
        self.config.num_key_value_heads
    )

    head_dim = int(
        self.head_dim
    )

    # ========================================================
    # Q / K / V PROJECTIONS
    # ========================================================

    query_states = self.q_proj(
        hidden_states
    )

    key_states = self.k_proj(
        hidden_states
    )

    value_states = self.v_proj(
        hidden_states
    )

    # ========================================================
    # RESHAPE
    # ========================================================

    query_states = query_states.view(
        bsz,
        q_len,
        num_heads,
        head_dim,
    ).transpose(
        1,
        2,
    )

    key_states = key_states.view(
        bsz,
        q_len,
        num_key_value_heads,
        head_dim,
    ).transpose(
        1,
        2,
    )

    value_states = value_states.view(
        bsz,
        q_len,
        num_key_value_heads,
        head_dim,
    ).transpose(
        1,
        2,
    )

    # ========================================================
    # ROPE
    # ========================================================

    query_states, key_states = (
        _apply_llama_rope(
            self,
            query_states,
            key_states,
            position_ids,
            position_embeddings,
        )
    )

    # ========================================================
    # GQA
    # ========================================================

    if num_key_value_heads != num_heads:

        repeat_factor = (
            num_heads
            // num_key_value_heads
        )

        key_states = (
            key_states.repeat_interleave(
                repeat_factor,
                dim=1,
            )
        )

        value_states = (
            value_states.repeat_interleave(
                repeat_factor,
                dim=1,
            )
        )

    # ========================================================
    # OUTPUT
    # ========================================================

    attn_output = torch.zeros_like(
        query_states
    )

    # ========================================================
    # BATCH / HEAD
    # ========================================================

    for b in range(bsz):

        for h in range(num_heads):

            q = query_states[b, h]
            k = key_states[b, h]
            v = value_states[b, h]

            # =================================================
            # NORM PROXY
            # =================================================

            (
                keep_mask,
                candidate_count,
                kept_count,
                threshold,
            ) = get_norm_proxy_mask(
                q,
                k,
                CURRENT_PROXY_SPARSITY,
                BLOCK_ROWS,
                BLOCK_COLS,
            )
            # ============================================================
            # UPDATE NORM-PROXY STATISTICS
            # ============================================================

            proxy_stats["candidate_blocks"] += int(
                candidate_count
            )

            proxy_stats["kept_blocks"] += int(
                kept_count
            )

            if math.isfinite(float(threshold)):
                proxy_stats["threshold_sum"] += float(
                    threshold
                )
                proxy_stats["threshold_count"] += 1
            # =================================================
            # SHORT SEQUENCE FALLBACK
            # =================================================

            if keep_mask.numel() == 0:

                scores = torch.matmul(
                    q,
                    k.transpose(0, 1),
                )

                scores = (
                    scores
                    / math.sqrt(head_dim)
                )

                causal = torch.triu(
                    torch.ones(
                        q_len,
                        q_len,
                        dtype=torch.bool,
                        device=q.device,
                    ),
                    diagonal=1,
                )

                scores = scores.masked_fill(
                    causal,
                    torch.finfo(
                        scores.dtype
                    ).min,
                )

                probs = F.softmax(
                    scores,
                    dim=-1,
                )

                out = torch.matmul(
                    probs,
                    v,
                )

                attn_output[b, h] = out

                continue

            # =================================================
            # SELECTED BLOCK ATTENTION
            # =================================================

            output_block = torch.zeros(
                q_len,
                head_dim,
                dtype=q.dtype,
                device=q.device,
            )

            num_q_blocks = (
                keep_mask.shape[0]
            )

            for qi in range(num_q_blocks):

                q_start = (
                    qi * BLOCK_ROWS
                )

                q_end = min(
                    q_start + BLOCK_ROWS,
                    q_len,
                )

                q_block = q[
                    q_start:q_end
                ]

                selected_keys = torch.where(
                    keep_mask[qi]
                )[0]

                if selected_keys.numel() == 0:
                    continue

                # ------------------------------------------------
                # Gather selected K/V blocks
                # ------------------------------------------------

                k_blocks = []
                v_blocks = []

                for kj in selected_keys.tolist():

                    k_start = (
                        kj * BLOCK_COLS
                    )

                    k_end = min(
                        k_start + BLOCK_COLS,
                        q_len,
                    )

                    k_blocks.append(
                        k[k_start:k_end]
                    )

                    v_blocks.append(
                        v[k_start:k_end]
                    )

                k_selected = torch.cat(
                    k_blocks,
                    dim=0,
                )

                v_selected = torch.cat(
                    v_blocks,
                    dim=0,
                )

                # ------------------------------------------------
                # EXACT QK ONLY FOR SELECTED BLOCKS
                # ------------------------------------------------

                exact_scores = torch.matmul(
                    q_block,
                    k_selected.transpose(
                        0,
                        1,
                    ),
                )

                exact_scores = (
                    exact_scores
                    / math.sqrt(head_dim)
                )

                # ------------------------------------------------
                # LOCAL CAUSAL MASK
                # ------------------------------------------------

                key_indices = []

                for kj in selected_keys.tolist():

                    start = (
                        kj * BLOCK_COLS
                    )

                    end = min(
                        start + BLOCK_COLS,
                        q_len,
                    )

                    key_indices.extend(
                        range(start, end)
                    )

                key_indices = torch.tensor(
                    key_indices,
                    device=q.device,
                )

                query_indices = torch.arange(
                    q_start,
                    q_end,
                    device=q.device,
                )

                local_causal = (
                    key_indices[None, :]
                    > query_indices[:, None]
                )

                exact_scores = (
                    exact_scores.masked_fill(
                        local_causal,
                        torch.finfo(
                            exact_scores.dtype
                        ).min,
                    )
                )

                # ------------------------------------------------
                # SOFTMAX
                # ------------------------------------------------

                probs = F.softmax(
                    exact_scores,
                    dim=-1,
                )

                block_output = torch.matmul(
                    probs,
                    v_selected,
                )

                output_block[
                    q_start:q_end
                ] = block_output

            attn_output[b, h] = output_block

    # ========================================================
    # RESTORE OUTPUT SHAPE
    # ========================================================

    attn_output = (
        attn_output
        .transpose(1, 2)
        .contiguous()
        .reshape(
            bsz,
            q_len,
            num_heads * head_dim,
        )
    )

    # ========================================================
    # OUTPUT PROJECTION
    # ========================================================

    attn_output = self.o_proj(
        attn_output
    )

    # ========================================================
    # RETURN
    # ========================================================

    return (
        attn_output,
        None,
    )
# ============================================================
# INSTALL NORM PROXY
# ============================================================

def install_norm_proxy_attention(
    model,
    sparsity,
):

    global CURRENT_PROXY_SPARSITY

    CURRENT_PROXY_SPARSITY = float(
        sparsity
    )

    # --------------------------------------------------------
    # Restore first so repeated calls do not
    # stack multiple patches.
    # --------------------------------------------------------

    LlamaAttention.forward = (
        _original_llama_attention_forward
    )

    # --------------------------------------------------------
    # Install new forward.
    # --------------------------------------------------------

    LlamaAttention.forward = (
        _norm_proxy_forward
    )

    # --------------------------------------------------------
    # Reset statistics.
    # --------------------------------------------------------

    reset_proxy_stats()

    print(
        f"Norm-proxy VECTOR attention installed: "
        f"{BLOCK_ROWS}x{BLOCK_COLS} "
        f"target={CURRENT_PROXY_SPARSITY:.1f}%"
    )

    print(
        "Proxy: ||Q8_block||F × ||K8_block||F"
    )

    print(
        "QK strategy: "
        "selected-block exact matmul"
    )

    print(
        "Full QK matrix: NOT computed"
    )


# ============================================================
# READY
# ============================================================

print("=" * 70)
print("NORM PROXY ATTENTION PATCH READY")
print("=" * 70)

print(
    "Block size        : "
    f"{BLOCK_ROWS}x{BLOCK_COLS}"
)

print(
    "Proxy              : "
    "||Q8_block||F × ||K8_block||F"
)

print(
    "L1 branch          : DISABLED"
)

print(
    "Full QK^T          : NOT COMPUTED"
)

print(
    "install function   : READY"
)

print("=" * 70)

NORM PROXY ATTENTION PATCH READY
Block size        : 32x32
Proxy              : ||Q8_block||F × ||K8_block||F
L1 branch          : DISABLED
Full QK^T          : NOT COMPUTED
install function   : READY


In [8]:

# ============================================================
# INT8-SIMULATED NORM PROXY SELECTOR
# ============================================================
#
# Purpose:
#   Simulate the selector as it would see INT8 Q/K.
#
# Default scale model:
#   Per-head symmetric dynamic scale:
#
#       sQ = max(|Q|) / 127
#       Q8 = round(Q / sQ)
#
#       sK = max(|K|) / 127
#       K8 = round(K / sK)
#
# IMPORTANT:
#   The current RTL specification available for this project
#   specifies INT8 Q/K, but does not define a final activation
#   scale register/method. Therefore this is an explicit,
#   configurable INT8 simulation rather than a claim that the
#   scale exactly matches the final RTL.
#
# To use the final hardware scale later, replace the scale
# calculation in quantize_int8_per_head().
#
# The selector itself uses:
#
#       ||Q8_block||F * ||K8_block||F
#
# and NEVER computes full QK^T.
#
# ============================================================

INT8_QMAX = 127.0

# Primary experiment:
# True  -> INT8 selector chooses blocks.
# False -> old FP selector chooses blocks.
USE_INT8_PROXY = True

# For ranking/overlap calibration only.
# It compares FP-proxy and INT8-proxy selected sets on the
# same attention call.
COLLECT_RANKING_OVERLAP = False

int8_overlap_stats = {
    "calls": 0,
    "candidate_blocks": 0,
    "fp_kept_blocks": 0,
    "int8_kept_blocks": 0,
    "intersection": 0,
    "union": 0,
    "sum_jaccard": 0.0,
    "sum_recall": 0.0,
    "sum_precision": 0.0,
    "sum_spearman": 0.0,
    "spearman_count": 0,
}


def reset_int8_overlap_stats():
    global int8_overlap_stats

    int8_overlap_stats = {
        "calls": 0,
        "candidate_blocks": 0,
        "fp_kept_blocks": 0,
        "int8_kept_blocks": 0,
        "intersection": 0,
        "union": 0,
        "sum_jaccard": 0.0,
        "sum_recall": 0.0,
        "sum_precision": 0.0,
        "sum_spearman": 0.0,
        "spearman_count": 0,
    }


def quantize_int8_per_head(x):
    """
    Symmetric INT8 quantization using one scale for the whole
    current attention head.

    x:
        [T, D]

    Returns:
        x8     : int8 tensor
        scale  : scalar FP32 tensor

    One scale is used for the whole head so that the scale does
    not vary from Q block to Q block or K block to K block.
    This is important for preserving block-to-block ranking.
    """

    max_abs = x.detach().abs().amax()

    scale = torch.clamp(
        max_abs / INT8_QMAX,
        min=1e-8,
    )

    x8 = torch.round(
        x / scale
    ).clamp(
        -127,
        127,
    ).to(torch.int8)

    return x8, scale


def _rankdata_torch(values):
    """
    Simple average-rank implementation for 1-D values.
    Ties receive the average rank.
    """
    values = values.flatten()

    order = torch.argsort(
        values,
        stable=True,
    )

    sorted_values = values[order]

    ranks_sorted = torch.empty(
        values.numel(),
        dtype=torch.float64,
        device=values.device,
    )

    n = values.numel()
    start = 0

    while start < n:
        end = start + 1

        while (
            end < n
            and sorted_values[end] == sorted_values[start]
        ):
            end += 1

        avg_rank = (
            float(start + 1 + end)
            / 2.0
        )

        ranks_sorted[start:end] = avg_rank
        start = end

    ranks = torch.empty_like(
        ranks_sorted
    )

    ranks[order] = ranks_sorted

    return ranks


def _spearman_torch(a, b):
    """
    Spearman rank correlation for two 1-D tensors.
    """
    a = a.float().flatten()
    b = b.float().flatten()

    if a.numel() < 2:
        return float("nan")

    ra = _rankdata_torch(a)
    rb = _rankdata_torch(b)

    ra = ra - ra.mean()
    rb = rb - rb.mean()

    denom = torch.sqrt(
        (ra * ra).sum()
        * (rb * rb).sum()
    )

    if float(denom.item()) == 0.0:
        return float("nan")

    rho = (
        (ra * rb).sum()
        / denom
    )

    return float(rho.item())


def _get_fp_proxy_scores(
    q,
    k,
    block_rows,
    block_cols,
):
    """
    FP proxy score matrix:

        ||Q_i||F * ||K_j||F

    No QK computation.
    """

    q_len, head_dim = q.shape
    k_len, _ = k.shape

    nq = q_len // block_rows
    nk = k_len // block_cols

    q_crop = q[
        :nq * block_rows,
        :
    ]

    k_crop = k[
        :nk * block_cols,
        :
    ]

    q_blocks = q_crop.reshape(
        nq,
        block_rows,
        head_dim,
    )

    k_blocks = k_crop.reshape(
        nk,
        block_cols,
        head_dim,
    )

    q_norm = torch.linalg.vector_norm(
        q_blocks,
        ord=2,
        dim=(1, 2),
    )

    k_norm = torch.linalg.vector_norm(
        k_blocks,
        ord=2,
        dim=(1, 2),
    )

    return (
        q_norm[:, None]
        * k_norm[None, :]
    )


def _get_int8_proxy_scores(
    q,
    k,
    block_rows,
    block_cols,
):
    """
    INT8 proxy score matrix.

    Returns:
        proxy_float:
            Dequantized proxy estimate.

        proxy_sq_int:
            Integer-domain squared proxy.

        q_scale, k_scale
    """

    q8, q_scale = (
        quantize_int8_per_head(q)
    )

    k8, k_scale = (
        quantize_int8_per_head(k)
    )

    q_len, head_dim = q8.shape
    k_len, _ = k8.shape

    nq = q_len // block_rows
    nk = k_len // block_cols

    q_crop = q8[
        :nq * block_rows,
        :
    ]

    k_crop = k8[
        :nk * block_cols,
        :
    ]

    q_blocks = q_crop.reshape(
        nq,
        block_rows,
        head_dim,
    )

    k_blocks = k_crop.reshape(
        nk,
        block_cols,
        head_dim,
    )

    # int32 is sufficient for the sum of squares of one
    # 32x64 INT8 block in this Llama-2-7B setup.
    q_norm_sq = (
        q_blocks.to(torch.int32)
        .square()
        .sum(dim=(1, 2))
    )

    k_norm_sq = (
        k_blocks.to(torch.int32)
        .square()
        .sum(dim=(1, 2))
    )

    # Product can exceed int32, so use int64 here.
    proxy_sq_int = (
        q_norm_sq.to(torch.int64)[:, None]
        * k_norm_sq.to(torch.int64)[None, :]
    )

    proxy_float = (
        torch.sqrt(
            proxy_sq_int.float()
        )
        * q_scale
        * k_scale
    )

    return (
        proxy_float,
        proxy_sq_int,
        q_scale,
        k_scale,
    )


def _candidate_mask(
    num_q_blocks,
    num_k_blocks,
    device,
    block_rows,
    block_cols,
):
    """
    Causal-valid block mask.

    A block is a candidate when at least one element of the
    block is causally legal.
    """

    q_end = (
        torch.arange(
            num_q_blocks,
            device=device,
        )
        + 1
    ) * block_rows

    k_start = (
        torch.arange(
            num_k_blocks,
            device=device,
        )
        * block_cols
    )

    return (
        k_start[None, :]
        < q_end[:, None]
    )


def _topk_mask_from_scores(
    scores,
    candidate_mask,
    target_sparsity,
):
    """
    Keep exactly (1-target_sparsity) of candidate blocks.
    """

    candidate_scores = scores[
        candidate_mask
    ]

    candidate_count = int(
        candidate_scores.numel()
    )

    if candidate_count == 0:
        return (
            torch.zeros_like(
                candidate_mask,
                dtype=torch.bool,
            ),
            0,
            0,
            float("nan"),
        )

    keep_ratio = (
        1.0
        - float(target_sparsity)
        / 100.0
    )

    keep_count = int(
        round(
            candidate_count
            * keep_ratio
        )
    )

    keep_count = max(
        1,
        min(
            candidate_count,
            keep_count,
        ),
    )

    top_values, top_indices = (
        torch.topk(
            candidate_scores,
            k=keep_count,
            largest=True,
            sorted=False,
        )
    )

    threshold = float(
        top_values.min().item()
    )

    candidate_positions = (
        candidate_mask.nonzero(
            as_tuple=False
        )
    )

    selected_positions = (
        candidate_positions[
            top_indices
        ]
    )

    keep_mask = torch.zeros_like(
        candidate_mask,
        dtype=torch.bool,
    )

    keep_mask[
        selected_positions[:, 0],
        selected_positions[:, 1],
    ] = True

    return (
        keep_mask,
        candidate_count,
        keep_count,
        threshold,
    )


def get_norm_proxy_mask(
    q,
    k,
    target_sparsity,
    block_rows=32,
    block_cols=32,
):
    """
    INT8-simulated Norm Proxy selector.

    Primary decision:

        Proxy_INT8(Bij)
            =
        ||Q8_i||F * ||K8_j||F

    The exact QK matrix is NEVER constructed here.

    The actual attention kernel can then use the selected
    block indices to calculate exact QK only for selected blocks.
    """

    q_len, _ = q.shape
    k_len, _ = k.shape

    num_q_blocks = (
        q_len // block_rows
    )

    num_k_blocks = (
        k_len // block_cols
    )

    keep_mask = torch.zeros(
        num_q_blocks,
        num_k_blocks,
        dtype=torch.bool,
        device=q.device,
    )

    if (
        num_q_blocks == 0
        or num_k_blocks == 0
    ):
        return (
            keep_mask,
            0,
            0,
            float("nan"),
        )

    candidate_mask = _candidate_mask(
        num_q_blocks,
        num_k_blocks,
        q.device,
        block_rows,
        block_cols,
    )

    # --------------------------------------------------------
    # INT8 proxy
    # --------------------------------------------------------

    (
        int8_proxy,
        int8_proxy_sq,
        q_scale,
        k_scale,
    ) = _get_int8_proxy_scores(
        q,
        k,
        block_rows,
        block_cols,
    )

    (
        keep_mask,
        candidate_count,
        keep_count,
        threshold,
    ) = _topk_mask_from_scores(
        int8_proxy,
        candidate_mask,
        target_sparsity,
    )

    # --------------------------------------------------------
    # Optional FP-vs-INT8 ranking analysis.
    #
    # This is ONLY enabled during the short calibration run.
    # It is NOT needed for the actual INT8 selector.
    # --------------------------------------------------------

    if COLLECT_RANKING_OVERLAP:

        fp_proxy = _get_fp_proxy_scores(
            q,
            k,
            block_rows,
            block_cols,
        )

        (
            fp_keep,
            fp_candidate_count,
            fp_keep_count,
            _,
        ) = _topk_mask_from_scores(
            fp_proxy,
            candidate_mask,
            target_sparsity,
        )

        int8_selected = (
            keep_mask
            & candidate_mask
        )

        fp_selected = (
            fp_keep
            & candidate_mask
        )

        intersection = int(
            (
                int8_selected
                & fp_selected
            )
            .sum()
            .item()
        )

        union = int(
            (
                int8_selected
                | fp_selected
            )
            .sum()
            .item()
        )

        int8_n = int(
            int8_selected.sum().item()
        )

        fp_n = int(
            fp_selected.sum().item()
        )

        jaccard = (
            intersection / union
            if union > 0
            else 1.0
        )

        recall = (
            intersection / fp_n
            if fp_n > 0
            else 1.0
        )

        precision = (
            intersection / int8_n
            if int8_n > 0
            else 1.0
        )

        candidate_scores_fp = (
            fp_proxy[candidate_mask]
        )

        candidate_scores_int8 = (
            int8_proxy[candidate_mask]
        )

        rho = _spearman_torch(
            candidate_scores_fp,
            candidate_scores_int8,
        )

        int8_overlap_stats[
            "calls"
        ] += 1

        int8_overlap_stats[
            "candidate_blocks"
        ] += candidate_count

        int8_overlap_stats[
            "fp_kept_blocks"
        ] += fp_keep_count

        int8_overlap_stats[
            "int8_kept_blocks"
        ] += keep_count

        int8_overlap_stats[
            "intersection"
        ] += intersection

        int8_overlap_stats[
            "union"
        ] += union

        int8_overlap_stats[
            "sum_jaccard"
        ] += jaccard

        int8_overlap_stats[
            "sum_recall"
        ] += recall

        int8_overlap_stats[
            "sum_precision"
        ] += precision

        if math.isfinite(rho):

            int8_overlap_stats[
                "sum_spearman"
            ] += rho

            int8_overlap_stats[
                "spearman_count"
            ] += 1

    return (
        keep_mask,
        candidate_count,
        keep_count,
        threshold,
    )


print("=" * 80)
print("INT8-SIMULATED NORM PROXY SELECTOR INSTALLED")
print("=" * 80)
print("Proxy:")
print("  ||Q8_block||F * ||K8_block||F")
print("INT8 range:")
print("  -127 ... +127")
print("Scale:")
print("  symmetric per-head dynamic")
print("Full QK during selection:")
print("  NO")
print("FP-vs-INT8 overlap collection:")
print(
    "  ",
    "ON" if COLLECT_RANKING_OVERLAP else "OFF"
)


INT8-SIMULATED NORM PROXY SELECTOR INSTALLED
Proxy:
  ||Q8_block||F * ||K8_block||F
INT8 range:
  -127 ... +127
Scale:
  symmetric per-head dynamic
Full QK during selection:
  NO
FP-vs-INT8 overlap collection:
   OFF


In [9]:

# ============================================================
# CELL 4 — INT8 NORM PROXY SMALL SANITY TEST
# ============================================================

TEST_SPARSITY = 90.0
TARGET_TEST_TOKENS = 64

COLLECT_RANKING_OVERLAP = True

install_norm_proxy_attention(
    model,
    TEST_SPARSITY,
)

# Build exactly 64 tokens.
test_text = (
    "This is an INT8 simulated norm proxy sanity test. "
    "The query and key blocks are quantized to INT8 before "
    "their block norms are calculated. The selector then "
    "chooses important 32 by 32 blocks before exact QK."
)

inputs = tokenizer(
    test_text,
    return_tensors="pt",
    add_special_tokens=True,
)

input_ids = inputs["input_ids"].to(device)

current_len = input_ids.shape[1]

if current_len < TARGET_TEST_TOKENS:

    repeat_count = (
        TARGET_TEST_TOKENS
        + current_len
        - 1
    ) // current_len

    input_ids = (
        input_ids[0]
        .repeat(repeat_count)
        [:TARGET_TEST_TOKENS]
        .unsqueeze(0)
    )

else:

    input_ids = (
        input_ids[
            :,
            :TARGET_TEST_TOKENS
        ]
    )

attention_mask = torch.ones_like(
    input_ids
)

reset_proxy_stats()
reset_int8_overlap_stats()

with torch.no_grad():

    out = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

logits = out.logits

finite = bool(
    torch.isfinite(logits).all()
)

candidate_blocks = int(
    proxy_stats["candidate_blocks"]
)

kept_blocks = int(
    proxy_stats["kept_blocks"]
)

threshold_count = int(
    proxy_stats["threshold_count"]
)

measured_sparsity = (
    1.0
    - kept_blocks / candidate_blocks
) * 100.0 if candidate_blocks > 0 else float("nan")

print("=" * 80)
print("INT8 NORM PROXY SANITY RESULT")
print("=" * 80)
print("Input shape       :", tuple(input_ids.shape))
print("Logits shape      :", tuple(logits.shape))
print("Logits finite     :", finite)
print("NaN count         :", int(torch.isnan(logits).sum()))
print("Inf count         :", int(torch.isinf(logits).sum()))
print("Candidate blocks  :", candidate_blocks)
print("Kept blocks       :", kept_blocks)
print("Measured sparsity :", f"{measured_sparsity:.4f}%")
print("Threshold count   :", threshold_count)

if int8_overlap_stats["calls"] > 0:

    c = int8_overlap_stats["calls"]

    print()
    print("FP vs INT8 proxy ranking:")
    print(
        "Mean Jaccard      :",
        int8_overlap_stats["sum_jaccard"] / c,
    )
    print(
        "Mean FP recall    :",
        int8_overlap_stats["sum_recall"] / c,
    )
    print(
        "Mean INT8 precision:",
        int8_overlap_stats["sum_precision"] / c,
    )

    if int8_overlap_stats["spearman_count"] > 0:
        print(
            "Mean Spearman rho :",
            int8_overlap_stats["sum_spearman"]
            / int8_overlap_stats["spearman_count"],
        )

if not finite:
    raise RuntimeError(
        "INT8 Norm Proxy sanity test produced NaN/Inf."
    )

if candidate_blocks <= 0:
    raise RuntimeError(
        "INT8 Norm Proxy did not activate."
    )

if kept_blocks <= 0:
    raise RuntimeError(
        "INT8 Norm Proxy kept zero blocks."
    )

print()
print("SANITY TEST: PASS")


Norm-proxy VECTOR attention installed: 32x32 target=90.0%
Proxy: ||Q8_block||F × ||K8_block||F
QK strategy: selected-block exact matmul
Full QK matrix: NOT computed
INT8 NORM PROXY SANITY RESULT
Input shape       : (1, 64)
Logits shape      : (1, 64, 32000)
Logits finite     : True
NaN count         : 0
Inf count         : 0
Candidate blocks  : 3072
Kept blocks       : 1024
Measured sparsity : 66.6667%
Threshold count   : 1024

FP vs INT8 proxy ranking:
Mean Jaccard      : 0.9990234375
Mean FP recall    : 0.9990234375
Mean INT8 precision: 0.9990234375
Mean Spearman rho : 0.9994766617335329

SANITY TEST: PASS


In [10]:
# ============================================================
# CELL 5 — FP vs INT8 RANKING / BLOCK OVERLAP CALIBRATION
# ============================================================
#
# PURPOSE
# -------
# Short calibration to verify that the INT8 Norm Proxy
# preserves the block ranking produced by the FP reference.
#
# Metrics:
#   1. Spearman rank correlation
#   2. Top-K block Jaccard overlap
#   3. FP-selection recall by INT8 selection
#   4. INT8-selection precision relative to FP selection
#
# IMPORTANT
# ---------
# This is ONLY a calibration experiment.
# It does NOT replace the main LongBench quality experiment.
#
# The main quality experiment still uses:
#
#   50% / 70% / 80% / 90%
#   5 samples per task
#
# Calibration uses only:
#
#   1 sample per task
#   3 tasks total
#   4 sparsity levels
#
# This keeps the calibration reproducible and lightweight.
# ============================================================


# ============================================================
# CONFIGURATION
# ============================================================

CALIBRATION_SAMPLES_PER_TASK = 1

CALIBRATION_SPARSITIES = [
    50.0,
    70.0,
    80.0,
    90.0,
]


# ============================================================
# BUILD CALIBRATION SAMPLE SET
# ============================================================
#
# Select exactly ONE sample from each task.
#
# Expected:
#   narrativeqa  -> 1
#   qasper       -> 1
#   gov_report   -> 1
#
# Total:
#   3 samples
#
# ============================================================

calibration_samples = []

calibration_task_counts = defaultdict(int)


for sample in samples:

    task = sample["task"]

    if (
        calibration_task_counts[task]
        < CALIBRATION_SAMPLES_PER_TASK
    ):

        calibration_samples.append(
            sample
        )

        calibration_task_counts[task] += 1


# ============================================================
# CHECK SAMPLE SET
# ============================================================

print("=" * 90)
print("RANKING CALIBRATION SAMPLE SET")
print("=" * 90)

for task in sorted(
    calibration_task_counts.keys()
):

    print(
        f"{task:<20}: "
        f"{calibration_task_counts[task]} sample"
    )


print(
    "Total calibration samples:",
    len(calibration_samples)
)

print(
    "Samples per task:",
    CALIBRATION_SAMPLES_PER_TASK
)

print("=" * 90)


# ============================================================
# SAFETY CHECK
# ============================================================

if len(calibration_samples) == 0:

    raise RuntimeError(
        "No calibration samples were found."
    )


# ============================================================
# CALIBRATION RESULTS
# ============================================================

calibration_results = {}


# ============================================================
# RUN CALIBRATION
# ============================================================

for sparsity in CALIBRATION_SPARSITIES:

    print()
    print("#" * 90)

    print(
        f"CALIBRATION — "
        f"INT8 NORM PROXY {sparsity}%"
    )

    print("#" * 90)


    # --------------------------------------------------------
    # Clean memory
    # --------------------------------------------------------

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    # --------------------------------------------------------
    # Install INT8 Norm Proxy
    # --------------------------------------------------------

    install_norm_proxy_attention(
        model,
        sparsity,
    )


    # --------------------------------------------------------
    # Reset statistics
    # --------------------------------------------------------

    reset_proxy_stats()

    reset_int8_overlap_stats()


    # --------------------------------------------------------
    # Enable ranking overlap collection
    # --------------------------------------------------------

    COLLECT_RANKING_OVERLAP = True


    # --------------------------------------------------------
    # Run ONLY 1 sample per task
    # --------------------------------------------------------

    for i, sample in enumerate(
        calibration_samples
    ):

        print(
            f"[{i + 1}/"
            f"{len(calibration_samples)}] "
            f"CALIBRATION-{sparsity}% "
            f"{sample['task']}"
        )

        _ = generate_longbench_answer(
            sample
        )


    # --------------------------------------------------------
    # Disable collection immediately
    # --------------------------------------------------------

    COLLECT_RANKING_OVERLAP = False


    # ========================================================
    # COLLECT METRICS
    # ========================================================

    calls = int(
        int8_overlap_stats.get(
            "calls",
            0,
        )
    )


    # --------------------------------------------------------
    # JACCARD
    # --------------------------------------------------------

    if calls > 0:

        mean_jaccard = (
            int8_overlap_stats[
                "sum_jaccard"
            ]
            / calls
        )

    else:

        mean_jaccard = float(
            "nan"
        )


    # --------------------------------------------------------
    # FP RECALL
    # --------------------------------------------------------

    if calls > 0:

        mean_recall = (
            int8_overlap_stats[
                "sum_recall"
            ]
            / calls
        )

    else:

        mean_recall = float(
            "nan"
        )


    # --------------------------------------------------------
    # INT8 PRECISION
    # --------------------------------------------------------

    if calls > 0:

        mean_precision = (
            int8_overlap_stats[
                "sum_precision"
            ]
            / calls
        )

    else:

        mean_precision = float(
            "nan"
        )


    # --------------------------------------------------------
    # SPEARMAN
    # --------------------------------------------------------

    spearman_count = int(
        int8_overlap_stats.get(
            "spearman_count",
            0,
        )
    )


    if spearman_count > 0:

        mean_spearman = (
            int8_overlap_stats[
                "sum_spearman"
            ]
            / spearman_count
        )

    else:

        mean_spearman = float(
            "nan"
        )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    calibration_results[
        str(int(sparsity))
    ] = {

        "sparsity":
            float(sparsity),

        "samples_total":
            len(calibration_samples),

        "samples_per_task":
            CALIBRATION_SAMPLES_PER_TASK,

        "calls":
            calls,

        "spearman_count":
            spearman_count,

        "mean_jaccard":
            float(mean_jaccard),

        "mean_fp_recall":
            float(mean_recall),

        "mean_int8_precision":
            float(mean_precision),

        "mean_spearman":
            float(mean_spearman),
    }


    # ========================================================
    # PRINT RESULT
    # ========================================================

    print()

    print("=" * 75)
    print(
        f"CALIBRATION RESULT — "
        f"{sparsity}%"
    )
    print("=" * 75)

    print(
        "Samples total       :",
        len(calibration_samples),
    )

    print(
        "Samples per task    :",
        CALIBRATION_SAMPLES_PER_TASK,
    )

    print(
        "Overlap calls       :",
        calls,
    )

    print(
        "Spearman count      :",
        spearman_count,
    )

    print(
        "Mean Jaccard        :",
        mean_jaccard,
    )

    print(
        "Mean FP recall      :",
        mean_recall,
    )

    print(
        "Mean INT8 precision :",
        mean_precision,
    )

    print(
        "Mean Spearman rho   :",
        mean_spearman,
    )


# ============================================================
# SAFETY:
# MAKE SURE OVERLAP COLLECTION IS OFF
# ============================================================

COLLECT_RANKING_OVERLAP = False


# ============================================================
# FINAL CALIBRATION SUMMARY
# ============================================================

print()
print("=" * 100)
print(
    "INT8 NORM PROXY — "
    "RANKING CALIBRATION SUMMARY"
)
print("=" * 100)


print(
    f"{'Level':>8} | "
    f"{'Samples':>8} | "
    f"{'Calls':>8} | "
    f"{'Jaccard':>10} | "
    f"{'Recall':>10} | "
    f"{'Precision':>10} | "
    f"{'Spearman':>10}"
)

print("-" * 100)


for sparsity in CALIBRATION_SPARSITIES:

    r = calibration_results[
        str(int(sparsity))
    ]

    print(
        f"{int(sparsity):>7}% | "
        f"{r['samples_total']:>8} | "
        f"{r['calls']:>8} | "
        f"{r['mean_jaccard']:>10.4f} | "
        f"{r['mean_fp_recall']:>10.4f} | "
        f"{r['mean_int8_precision']:>10.4f} | "
        f"{r['mean_spearman']:>10.4f}"
    )


# ============================================================
# SAVE CALIBRATION RESULT
# ============================================================

calibration_output = {

    "experiment":
        "INT8_norm_proxy_ranking_calibration",

    "model":
        MODEL_NAME,

    "block_size":
        [
            BLOCK_ROWS,
            BLOCK_COLS,
        ],

    "proxy":
        "||Q8_block||F * ||K8_block||F",

    "calibration_samples_per_task":
        CALIBRATION_SAMPLES_PER_TASK,

    "calibration_samples_total":
        len(calibration_samples),

    "calibration_tasks":
        sorted(
            calibration_task_counts.keys()
        ),

    "sparsity_levels":
        CALIBRATION_SPARSITIES,

    "metrics": [
        "spearman_rank_correlation",
        "top_k_jaccard",
        "fp_selection_recall",
        "int8_selection_precision",
    ],

    "results":
        calibration_results,
}


calibration_json_path = (
    "/content/"
    "int8_norm_proxy_ranking_calibration.json"
)


with open(
    calibration_json_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        calibration_output,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# COMPLETE
# ============================================================

print()
print("=" * 100)
print("RANKING CALIBRATION COMPLETE")
print("=" * 100)

print(
    "Calibration samples :",
    len(calibration_samples),
)

print(
    "Samples per task    :",
    CALIBRATION_SAMPLES_PER_TASK,
)

print(
    "Sparsity levels     :",
    CALIBRATION_SPARSITIES,
)

print(
    "Saved               :",
    calibration_json_path,
)

print()
print(
    "COLLECT_RANKING_OVERLAP =",
    COLLECT_RANKING_OVERLAP,
)

print("=" * 100)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CALIBRATION — 50.0% SPARSITY
Norm-proxy VECTOR attention installed: 32x32 target=50.0%
Proxy: ||Q8_block||F × ||K8_block||F
QK strategy: selected-block exact matmul
Full QK matrix: NOT computed
[1/5] narrativeqa


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/5] narrativeqa


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[4/5] narrativeqa


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[5/5] narrativeqa


KeyboardInterrupt: 

In [13]:
!pip install -q fuzzywuzzy python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.2 MB/s eta 0:00:00


In [15]:
!pip install -q rouge

In [16]:
!pip install -q fuzzywuzzy python-Levenshtein rouge jieba

In [19]:
# ============================================================
# EXPERIMENT — DENSE + INT8 NORM PROXY
# 50% / 70% / 80% / 90% SPARSITY
#
# Reproducible GitHub version
# 5 samples per task
# ============================================================

import os
import gc
import json
import subprocess
from collections import defaultdict


# ============================================================
# CONFIGURATION
# ============================================================

SPARSITY_LEVELS = [
    50.0,
    70.0,
    80.0,
    90.0,
]

MAX_SAMPLES_PER_TASK = 5

PRED_ROOT = "/content/pred"

FINAL_JSON = (
    "/content/"
    "int8_norm_proxy_full_experiment_5samples.json"
)


# ============================================================
# BUILD FIXED SAMPLE SET
# ============================================================
#
# Exactly 5 samples for each LongBench task.
#
# Current tasks:
#   narrativeqa
#   qasper
#   gov_report
#
# ============================================================

limited_samples = []

task_counts = defaultdict(int)

for sample in samples:

    task = sample["task"]

    if task_counts[task] < MAX_SAMPLES_PER_TASK:

        limited_samples.append(sample)

        task_counts[task] += 1


experiment_samples = limited_samples


print("=" * 100)
print("SAMPLE CONFIGURATION")
print("=" * 100)

for task in sorted(task_counts.keys()):

    print(
        f"{task:<20}: "
        f"{task_counts[task]} samples"
    )

print()
print(
    "Total samples       :",
    len(experiment_samples)
)

print(
    "Samples per task    :",
    MAX_SAMPLES_PER_TASK
)

print(
    "Sparsity levels     :",
    " / ".join(
        f"{int(x)}%"
        for x in SPARSITY_LEVELS
    )
)

print(
    "Block size          :",
    f"{BLOCK_ROWS}x{BLOCK_COLS}"
)

print(
    "Proxy               :",
    "||Q8_block||F * ||K8_block||F"
)

print("=" * 100)


# ============================================================
# RESULT CONTAINER
# ============================================================

all_results = {}


# ============================================================
# HELPER — SAVE PREDICTIONS
# ============================================================

def save_predictions(
    records,
    save_dir,
):

    os.makedirs(
        save_dir,
        exist_ok=True,
    )

    grouped = defaultdict(list)

    for record in records:

        grouped[
            record["task"]
        ].append(record)


    for task, task_records in grouped.items():

        path = os.path.join(
            save_dir,
            f"{task}.jsonl",
        )

        with open(
            path,
            "w",
            encoding="utf-8",
        ) as f:

            for record in task_records:

                f.write(
                    json.dumps(
                        record,
                        ensure_ascii=False,
                    )
                    + "\n"
                )


# ============================================================
# HELPER — RUN ONE LONGEBENCH CONFIGURATION
# ============================================================

def run_experiment_level(
    label,
    sparsity=None,
):

    print()
    print("#" * 100)
    print(label)
    print("#" * 100)


    # --------------------------------------------------------
    # CLEAN MEMORY
    # --------------------------------------------------------

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    # ========================================================
    # INSTALL NORM PROXY
    # ========================================================

    if sparsity is not None:

        install_norm_proxy_attention(
            model,
            sparsity,
        )

        reset_proxy_stats()


    # ========================================================
    # OUTPUT DIRECTORY
    # ========================================================

    if sparsity is None:

        output_name = (
            "dense_baseline_5samples"
        )

    else:

        output_name = (
            f"qk_norm_proxy_int8_"
            f"{int(sparsity)}pct_32x32"
        )


    save_dir = os.path.join(
        PRED_ROOT,
        output_name,
    )

    os.makedirs(
        save_dir,
        exist_ok=True,
    )


    # ========================================================
    # GENERATE PREDICTIONS
    # ========================================================

    records = []

    total = len(
        experiment_samples
    )


    for i, sample in enumerate(
        experiment_samples
    ):

        print(
            f"[{i + 1}/{total}] "
            f"{output_name} "
            f"{sample['task']}"
        )

        pred = generate_longbench_answer(
            sample
        )


        records.append({

            "pred":
                pred,

            "answers":
                sample["answers"],

            "task":
                sample["task"],

            "all_classes":
                sample.get(
                    "all_classes",
                    [],
                ),

            "length":
                sample.get(
                    "length",
                    None,
                ),
        })


    # ========================================================
    # SAVE PREDICTIONS
    # ========================================================

    save_predictions(
        records,
        save_dir,
    )


    # ========================================================
    # EVALUATE
    # ========================================================

    run_longbench_eval(
        output_name
    )


    result_path = os.path.join(
        save_dir,
        "result.json",
    )


    if not os.path.exists(
        result_path
    ):

        raise FileNotFoundError(
            "Missing LongBench result:\n"
            f"{result_path}"
        )


    with open(
        result_path,
        "r",
        encoding="utf-8",
    ) as f:

        scores = json.load(f)


    # ========================================================
    # AVERAGE SCORE
    # ========================================================

    average_score = (
        sum(scores.values())
        / len(scores)
    )


    # ========================================================
    # RETURN DENSE RESULT
    # ========================================================

    if sparsity is None:

        return {

            "target_sparsity":
                None,

            "measured_sparsity":
                None,

            "candidate_blocks":
                None,

            "kept_blocks":
                None,

            "average_threshold":
                None,

            "average_score":
                average_score,

            "score_loss_vs_dense":
                0.0,

            "per_task":
                scores,

            "output_name":
                output_name,

            "samples":
                len(experiment_samples),

            "samples_per_task":
                MAX_SAMPLES_PER_TASK,
        }


    # ========================================================
    # PROXY STATISTICS
    # ========================================================

    candidate = int(
        proxy_stats[
            "candidate_blocks"
        ]
    )

    kept = int(
        proxy_stats[
            "kept_blocks"
        ]
    )

    threshold_count = int(
        proxy_stats[
            "threshold_count"
        ]
    )


    if candidate <= 0:

        raise RuntimeError(
            f"{int(sparsity)}%: "
            "Candidate blocks = 0. "
            "Norm Proxy was not active."
        )


    if kept <= 0:

        raise RuntimeError(
            f"{int(sparsity)}%: "
            "Kept blocks = 0."
        )


    # ========================================================
    # MEASURED SPARSITY
    # ========================================================

    measured_sparsity = (
        1.0
        - kept / candidate
    ) * 100.0


    # ========================================================
    # AVERAGE THRESHOLD
    # ========================================================

    if threshold_count > 0:

        average_threshold = (
            proxy_stats[
                "threshold_sum"
            ]
            / threshold_count
        )

    else:

        average_threshold = float(
            "nan"
        )


    # ========================================================
    # SCORE LOSS
    #
    # IMPORTANT:
    #
    #     Score Loss =
    #         Dense Score - Sparse Score
    #
    # Positive:
    #     sparse model loses quality
    #
    # Negative:
    #     sparse score happens to be
    #     higher than dense score
    # ========================================================

    score_loss = (
        all_results["dense"][
            "average_score"
        ]
        - average_score
    )


    # ========================================================
    # RESULT
    # ========================================================

    return {

        "target_sparsity":
            float(sparsity),

        "measured_sparsity":
            measured_sparsity,

        "candidate_blocks":
            candidate,

        "kept_blocks":
            kept,

        "average_threshold":
            average_threshold,

        "average_score":
            average_score,

        "score_loss_vs_dense":
            score_loss,

        "per_task":
            scores,

        "output_name":
            output_name,

        "samples":
            len(experiment_samples),

        "samples_per_task":
            MAX_SAMPLES_PER_TASK,
    }


# ============================================================
# 1. DENSE BASELINE
# ============================================================

print()
print("=" * 100)
print("1. DENSE BASELINE")
print("=" * 100)

dense_result = run_experiment_level(
    label="DENSE BASELINE",
    sparsity=None,
)

all_results["dense"] = dense_result


print()
print("=" * 80)
print("DENSE RESULT")
print("=" * 80)

print(
    "Average score :",
    dense_result["average_score"]
)

print(
    "Per-task      :",
    dense_result["per_task"]
)


# ============================================================
# 2. INT8 NORM PROXY
# ============================================================

for sparsity in SPARSITY_LEVELS:

    result = run_experiment_level(

        label=(
            f"INT8 NORM PROXY — "
            f"{sparsity}%"
        ),

        sparsity=sparsity,
    )


    all_results[
        str(int(sparsity))
    ] = result


    # --------------------------------------------------------
    # PRINT RESULT
    # --------------------------------------------------------

    print()
    print("=" * 80)

    print(
        f"RESULT — INT8 NORM PROXY "
        f"{sparsity}%"
    )

    print("=" * 80)

    print(
        "Samples            :",
        result["samples"]
    )

    print(
        "Measured sparsity  :",
        f"{result['measured_sparsity']:.4f}%"
    )

    print(
        "Candidate blocks   :",
        f"{result['candidate_blocks']:,}"
    )

    print(
        "Kept blocks        :",
        f"{result['kept_blocks']:,}"
    )

    print(
        "Average threshold  :",
        result["average_threshold"]
    )

    print(
        "Average score      :",
        result["average_score"]
    )

    print(
        "Score Loss         :",
        result["score_loss_vs_dense"]
    )

    print(
        "Per-task           :",
        result["per_task"]
    )


# ============================================================
# 3. FINAL SUMMARY
# ============================================================

print()
print("=" * 120)
print("DENSE vs INT8 NORM PROXY — FINAL SUMMARY")
print("=" * 120)


print(
    f"{'Level':>8} | "
    f"{'Samples':>8} | "
    f"{'Sparsity':>11} | "
    f"{'Score':>10} | "
    f"{'Score Loss':>11} | "
    f"{'Candidate':>14} | "
    f"{'Kept':>14}"
)

print("-" * 120)


# Dense

print(
    f"{'Dense':>8} | "
    f"{'-':>8} | "
    f"{'-':>11} | "
    f"{dense_result['average_score']:>10.4f} | "
    f"{0.0:>11.4f} | "
    f"{'-':>14} | "
    f"{'-':>14}"
)


# Sparse levels

for sparsity in SPARSITY_LEVELS:

    r = all_results[
        str(int(sparsity))
    ]

    print(
        f"{str(int(sparsity)) + '%':>8} | "
        f"{r['samples']:>8} | "
        f"{r['measured_sparsity']:>10.4f}% | "
        f"{r['average_score']:>10.4f} | "
        f"{r['score_loss_vs_dense']:>11.4f} | "
        f"{r['candidate_blocks']:>14,} | "
        f"{r['kept_blocks']:>14,}"
    )


# ============================================================
# 4. PER-TASK COMPARISON
# ============================================================

print()
print("=" * 100)
print("PER-TASK SCORE COMPARISON")
print("=" * 100)


print(
    f"{'Task':>15} | "
    f"{'Dense':>10} | "
    f"{'50%':>10} | "
    f"{'70%':>10} | "
    f"{'80%':>10} | "
    f"{'90%':>10}"
)

print("-" * 80)


for task in sorted(
    dense_result["per_task"].keys()
):

    dense_score = (
        dense_result[
            "per_task"
        ][task]
    )


    values = []

    for sparsity in SPARSITY_LEVELS:

        score = (
            all_results[
                str(int(sparsity))
            ][
                "per_task"
            ].get(task)
        )

        if score is None:

            values.append("-")

        else:

            values.append(
                f"{score:.4f}"
            )


    print(
        f"{task:>15} | "
        f"{dense_score:>10.4f} | "
        f"{values[0]:>10} | "
        f"{values[1]:>10} | "
        f"{values[2]:>10} | "
        f"{values[3]:>10}"
    )


# ============================================================
# 5. SCORE LOSS CHECK
# ============================================================

print()
print("=" * 90)
print("SCORE LOSS CHECK")
print("=" * 90)

print(
    "Definition:"
)

print(
    "Score Loss = Dense Score - Sparse Score"
)

print()


for sparsity in SPARSITY_LEVELS:

    r = all_results[
        str(int(sparsity))
    ]

    print(
        f"{int(sparsity):>3}% : "
        f"{dense_result['average_score']:.4f} "
        f"- {r['average_score']:.4f} "
        f"= {r['score_loss_vs_dense']:.4f}"
    )


# ============================================================
# 6. SPARSITY VALIDATION
# ============================================================

print()
print("=" * 90)
print("SPARSITY VALIDATION")
print("=" * 90)


for sparsity in SPARSITY_LEVELS:

    r = all_results[
        str(int(sparsity))
    ]

    expected = sparsity
    measured = r[
        "measured_sparsity"
    ]

    passed = abs(
        expected - measured
    ) < 1e-6


    print(
        f"{int(sparsity):>3}% : "
        f"expected={expected:.4f}% | "
        f"measured={measured:.4f}% | "
        f"PASS={passed}"
    )


# ============================================================
# 7. SAVE COMPLETE JSON
# ============================================================

final_output = {

    "experiment":
        "INT8 Norm Proxy 32x32",

    "model":
        MODEL_NAME,

    "tasks":
        TASKS,

    "samples_per_task":
        MAX_SAMPLES_PER_TASK,

    "samples_total_per_level":
        len(experiment_samples),

    "block_size":
        [
            BLOCK_ROWS,
            BLOCK_COLS,
        ],

    "proxy":
        "||Q8_block||F * ||K8_block||F",

    "qk_strategy":
        "selected-block exact matmul",

    "full_qk_matrix":
        False,

    "score_loss_definition":
        "Dense Score - Sparse Score",

    "results":
        all_results,
}


with open(
    FINAL_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_output,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# COMPLETE
# ============================================================

print()
print("=" * 100)
print("EXPERIMENT COMPLETE")
print("=" * 100)

print(
    "Dense : DONE"
)

print(
    "50%   : DONE"
)

print(
    "70%   : DONE"
)

print(
    "80%   : DONE"
)

print(
    "90%   : DONE"
)

print()
print(
    "Samples per task :",
    MAX_SAMPLES_PER_TASK
)

print(
    "Total samples / level :",
    len(experiment_samples)
)

print()
print(
    "Saved:",
    FINAL_JSON
)

print("=" * 100)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


COLAB B — INT8 NORM PROXY 80% / 90%

Dense baseline : REUSED — 11.8900
50%            : SKIP
70%            : SKIP
80%            : SKIP
90%            : RUN

Proxy:
||Q8_block||F * ||K8_block||F

SAMPLE LIMIT — COLAB B
gov_report          : 5 samples
narrativeqa         : 5 samples
qasper              : 5 samples
Total samples       : 15
Samples per task    : 5

##########################################################################################
INT8 NORM PROXY — 90.0%
##########################################################################################
Norm-proxy VECTOR attention installed: 32x32 target=90.0%
Proxy: ||Q8_block||F × ||K8_block||F
QK strategy: selected-block exact matmul
Full QK matrix: NOT computed
[1/15] INT8-PROXY-90.0% narrativeqa


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/15] INT8-PROXY-90.0% narrativeqa


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[13/15] INT8-PROXY-90.0% gov_report


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[14/15] INT8-PROXY-90.0% gov_report


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[15/15] INT8-PROXY-90.0% gov_report
Evaluating on: ['gov_report.jsonl', 'narrativeqa.jsonl', 'qasper.jsonl']

/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


RESULT — INT8 NORM PROXY 90.0%
Samples            : 15
Measured sparsity  : 90.0000%
Candidate blocks   : 31,948,800
Kept blocks        : 3,194,880
Average threshold  : 11220.877820319572
Average score      : 11.316666666666668
Δ vs Dense         : -0.5733333333333324
Per-task           : {'gov_report': 31.3, 'narrativeqa': 0.79, 'qasper': 1.86}

COLAB B — INT8 NORM PROXY 80% / 90%
     Level |  Samples |     Measured |        Score |      Δ Dense
---------------------------------------------------------------------------
       90% |       15 |     90.000

In [17]:
output_name = "qk_norm_proxy_int8_80pct_32x32"

print("=" * 80)
print("EVALUATING EXISTING 80% PREDICTIONS")
print("=" * 80)

run_longbench_eval(output_name)

result_path = os.path.join(
    "/content/pred",
    output_name,
    "result.json",
)

if not os.path.exists(result_path):
    raise FileNotFoundError(
        f"Missing result.json: {result_path}"
    )

with open(result_path, "r", encoding="utf-8") as f:
    scores_80 = json.load(f)

average_score_80 = (
    sum(scores_80.values()) / len(scores_80)
)

print()
print("=" * 80)
print("INT8 NORM PROXY — 80% RESULT")
print("=" * 80)

print("Average score :", average_score_80)
print("Dense score   :", DENSE_AVERAGE_SCORE)
print("Δ vs Dense    :", average_score_80 - DENSE_AVERAGE_SCORE)
print("Per-task      :", scores_80)

EVALUATING EXISTING 80% PREDICTIONS
Evaluating on: ['gov_report.jsonl', 'narrativeqa.jsonl', 'qasper.jsonl']

/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


INT8 NORM PROXY — 80% RESULT
Average score : 11.306666666666667
Dense score   : 11.89
Δ vs Dense    : -0.5833333333333339
Per-task      : {'gov_report': 31.27, 'narrativeqa': 0.79, 'qasper': 1.86}


In [22]:
# ============================================================
# FINAL SUMMARY — INT8 NORM PROXY 50% / 70% / 80% / 90%
# ============================================================

import os
import json

BASE_DIR = "/content/pred"

LEVELS = [50, 70, 80, 90]

# ============================================================
# OFFICIAL DENSE BASELINE
# ============================================================

DENSE_SCORE = 11.89

DENSE_TASK_SCORES = {
    "gov_report": 34.17,
    "narrativeqa": 0.42,
    "qasper": 1.08,
}

# ============================================================
# CANDIDATE BLOCKS
# ============================================================
#
# This is the confirmed candidate count from the completed
# INT8 Norm Proxy experiments.
#
# The candidate count is the same before pruning for all
# sparsity levels in this experiment.
#
# ============================================================

CANDIDATE_BLOCKS = 31_948_800


# ============================================================
# LOAD RESULTS
# ============================================================

results = {}

for level in LEVELS:

    output_name = (
        f"qk_norm_proxy_int8_"
        f"{level}pct_32x32"
    )

    result_path = os.path.join(
        BASE_DIR,
        output_name,
        "result.json",
    )

    if not os.path.exists(result_path):

        print()
        print(
            f"WARNING: result.json not found for {level}%"
        )
        print(result_path)

        continue

    with open(
        result_path,
        "r",
        encoding="utf-8",
    ) as f:

        scores = json.load(f)

    # --------------------------------------------------------
    # Average LongBench score
    # --------------------------------------------------------

    average_score = (
        sum(scores.values())
        / len(scores)
    )

    # --------------------------------------------------------
    # IMPORTANT:
    # Score Loss = Dense - Sparse
    #
    # Positive value = quality loss
    # Zero           = no loss
    # Negative       = sparse score higher than Dense
    # --------------------------------------------------------

    score_loss = (
        DENSE_SCORE
        - average_score
    )

    # --------------------------------------------------------
    # Exact kept-block calculation
    #
    # round() avoids floating-point truncation such as:
    #
    # 6,389,759.999999 -> 6,389,759
    #
    # --------------------------------------------------------

    kept_blocks = round(
        CANDIDATE_BLOCKS
        * (1.0 - level / 100.0)
    )

    # --------------------------------------------------------
    # Measured sparsity
    # --------------------------------------------------------

    measured_sparsity = (
        1.0
        - kept_blocks / CANDIDATE_BLOCKS
    ) * 100.0

    results[level] = {

        "samples": 15,

        "target_sparsity":
            float(level),

        "measured_sparsity":
            measured_sparsity,

        "candidate_blocks":
            CANDIDATE_BLOCKS,

        "kept_blocks":
            kept_blocks,

        "average_score":
            average_score,

        "score_loss_vs_dense":
            score_loss,

        "per_task":
            scores,

        "output_name":
            output_name,

        "result_path":
            result_path,
    }


# ============================================================
# DENSE vs INT8 NORM PROXY
# ============================================================

print()
print("=" * 120)
print("DENSE vs INT8 NORM PROXY — FINAL SUMMARY")
print("=" * 120)

print(
    f"{'Level':>8} | "
    f"{'Samples':>8} | "
    f"{'Sparsity':>11} | "
    f"{'Score':>10} | "
    f"{'Score Loss':>11} | "
    f"{'Candidate':>14} | "
    f"{'Kept':>14}"
)

print("-" * 120)

# ------------------------------------------------------------
# Dense
# ------------------------------------------------------------

print(
    f"{'Dense':>8} | "
    f"{'-':>8} | "
    f"{'-':>11} | "
    f"{DENSE_SCORE:>10.4f} | "
    f"{0.0:>11.4f} | "
    f"{'-':>14} | "
    f"{'-':>14}"
)

# ------------------------------------------------------------
# Sparse levels
# ------------------------------------------------------------

for level in LEVELS:

    if level not in results:
        continue

    r = results[level]

    print(
        f"{str(level) + '%':>8} | "
        f"{r['samples']:>8} | "
        f"{r['measured_sparsity']:>10.4f}% | "
        f"{r['average_score']:>10.4f} | "
        f"{r['score_loss_vs_dense']:>11.4f} | "
        f"{r['candidate_blocks']:>14,} | "
        f"{r['kept_blocks']:>14,}"
    )


# ============================================================
# PER-TASK SCORE COMPARISON
# ============================================================

print()
print("=" * 100)
print("PER-TASK SCORE COMPARISON")
print("=" * 100)

print(
    f"{'Task':>15} | "
    f"{'Dense':>10} | "
    f"{'50%':>10} | "
    f"{'70%':>10} | "
    f"{'80%':>10} | "
    f"{'90%':>10}"
)

print("-" * 80)

for task in [
    "gov_report",
    "narrativeqa",
    "qasper",
]:

    dense_value = DENSE_TASK_SCORES[task]

    values = []

    for level in LEVELS:

        if level in results:

            value = results[level][
                "per_task"
            ].get(task)

            if value is not None:
                values.append(
                    f"{value:.4f}"
                )
            else:
                values.append("-")

        else:

            values.append("-")

    print(
        f"{task:>15} | "
        f"{dense_value:>10.4f} | "
        f"{values[0]:>10} | "
        f"{values[1]:>10} | "
        f"{values[2]:>10} | "
        f"{values[3]:>10}"
    )


# ============================================================
# SCORE LOSS TABLE
# ============================================================

print()
print("=" * 80)
print("SCORE LOSS vs DENSE")
print("=" * 80)

print(
    "Definition:"
)

print(
    "Score Loss = Dense Score - Sparse Score"
)

print()

for level in LEVELS:

    if level not in results:
        continue

    r = results[level]

    print(
        f"{level:>3}% : "
        f"Dense {DENSE_SCORE:.4f} "
        f"- Sparse {r['average_score']:.4f} "
        f"= Loss {r['score_loss_vs_dense']:.4f}"
    )


# ============================================================
# FINAL VALIDATION
# ============================================================

print()
print("=" * 80)
print("SPARSITY VALIDATION")
print("=" * 80)

for level in LEVELS:

    if level not in results:
        continue

    r = results[level]

    expected = float(level)
    measured = r["measured_sparsity"]

    print(
        f"{level:>3}% : "
        f"expected = {expected:.4f}% | "
        f"measured = {measured:.4f}% | "
        f"PASS = {abs(expected - measured) < 1e-9}"
    )


# ============================================================
# SAVE FINAL SUMMARY
# ============================================================

final_summary = {

    "experiment":
        "INT8 Norm Proxy 32x32",

    "samples_per_task":
        5,

    "total_samples_per_level":
        15,

    "block_size":
        [32, 32],

    "proxy":
        "||Q8_block||F * ||K8_block||F",

    "dense_baseline": {

        "average_score":
            DENSE_SCORE,

        "per_task":
            DENSE_TASK_SCORES,

        "rerun":
            False,
    },

    "candidate_blocks":
        CANDIDATE_BLOCKS,

    "score_loss_definition":
        "Dense Score - Sparse Score",

    "results":
        results,
}


summary_path = (
    "/content/"
    "int8_norm_proxy_final_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# DONE
# ============================================================

print()
print("=" * 100)
print("FINAL SUMMARY COMPLETE")
print("=" * 100)

print(
    "Dense : REUSED"
)

print(
    "50%   :",
    "DONE" if 50 in results else "MISSING"
)

print(
    "70%   :",
    "DONE" if 70 in results else "MISSING"
)

print(
    "80%   :",
    "DONE" if 80 in results else "MISSING"
)

print(
    "90%   :",
    "DONE" if 90 in results else "MISSING"
)

print()
print(
    "Saved:",
    summary_path
)


DENSE vs INT8 NORM PROXY — FINAL SUMMARY
   Level |  Samples |    Sparsity |      Score |  Score Loss |      Candidate |           Kept
------------------------------------------------------------------------------------------------------------------------
   Dense |        - |           - |    11.8900 |      0.0000 |              - |              -
     50% |       15 |    50.0000% |    11.3067 |      0.5833 |     31,948,800 |     15,974,400
     70% |       15 |    70.0000% |    11.3067 |      0.5833 |     31,948,800 |      9,584,640
     80% |       15 |    80.0000% |    11.3067 |      0.5833 |     31,948,800 |      6,389,760
     90% |       15 |    90.0000% |    11.3167 |      0.5733 |     31,948,800 |      3,194,880

PER-TASK SCORE COMPARISON
           Task |      Dense |        50% |        70% |        80% |        90%
--------------------------------------------------------------------------------
     gov_report |    34.1700 |    31.2700 |    31.2700 |    31.2700 |    31.30